# 🚀 Tool Upload Video lên YouTube từ Google Drive
Công cụ này giúp bạn chuyển trực tiếp video từ Google Drive sang YouTube cực nhanh chỉ với một đường link.

**Hướng dẫn:**
1. Chạy từng ô (cell) từ trên xuống dưới bằng cách bấm nút ▶️ bên trái mỗi ô.
2. Đảm bảo bạn đã có file `client_secrets.json` từ Google Cloud Console.

In [ ]:
!pip install --upgrade -q google-api-python-client google-auth-oauthlib google-auth-httplib2 httplib2

In [ ]:
from google.colab import files
import os

print("Vui lòng tải lên file client_secrets.json của bạn (Lấy từ Google Cloud Console)")
if os.path.exists('client_secrets.json'):
    print("✅ File client_secrets.json đã tồn tại. Nếu muốn đổi file khác, hãy xóa nó trước.")
else:
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn != 'client_secrets.json':
            os.rename(fn, 'client_secrets.json')
    print("✅ Đã tải lên thành công file cấu hình API.")

In [ ]:
#@title ⚙️ Cấu hình Video
#@markdown Dán đường link Google Drive của video vào đây. (Tiêu đề mặc định là tên file, trạng thái mặc định là Riêng tư)

DRIVE_LINK = "https://drive.google.com/file/d/1QDDcjThJmiJGT7cl8RvRPfEWFzcAgaS3/view?usp=sharing" #@param {type:"string"}
PRIVACY_STATUS = "private" #@param ["public", "private", "unlisted"]
TITLE = "" #@param {type:"string"}
DESCRIPTION = "" #@param {type:"string"}
TAGS = "" #@param {type:"string"}

In [ ]:
import os
import re
import json
import io
from google.colab import auth
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

# Bỏ qua lỗi bắt buộc HTTPS khi dùng localhost (Fix InsecureTransportError)
os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

# 1. Xử lý link Google Drive và tải file
def get_drive_id(url):
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    match = re.search(r'id=([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    return url.strip()

file_id = get_drive_id(DRIVE_LINK)
if not file_id:
    raise ValueError("❌ Link Google Drive không hợp lệ!")

print("⏳ Đang yêu cầu quyền truy cập Google Drive để lấy video...")
auth.authenticate_user() # Hiển thị popup cấp quyền Colab
drive_service = build('drive', 'v3')

try:
    file_info = drive_service.files().get(fileId=file_id, fields='name').execute()
except Exception as e:
    raise Exception("❌ Không thể truy cập file này! Hãy chắc chắn link đúng và bạn có quyền xem file.")

file_name = file_info.get('name', 'video.mp4')
FILE_PATH = f"/content/{file_name}"

if not os.path.exists(FILE_PATH):
    print(f"⏳ Đang tải '{file_name}' từ Drive xuống Colab (tốc độ siêu nhanh)...")
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.FileIO(FILE_PATH, 'wb')
    downloader = MediaIoBaseDownload(fh, request, chunksize=1024*1024*50)
    done = False
    while done is False:
        status, done = downloader.next_chunk()
        if status:
            print(f"\rĐã tải được {int(status.progress() * 100)}%...", end="")
    print("\n✅ Tải xong video!")
else:
    print(f"✅ Video '{file_name}' đã có sẵn trong Colab.")

# 2. Cấu hình tiêu đề mặc định
video_title = TITLE.strip()
if not video_title:
    video_title = os.path.splitext(file_name)[0]

video_tags = [tag.strip() for tag in TAGS.split(',')] if TAGS.strip() else []

# 3. Xác thực YouTube API
if not os.path.exists('client_secrets.json'):
    raise FileNotFoundError("❌ Không tìm thấy file client_secrets.json! Vui lòng chạy bước tải file lên ở trên.")

with open('client_secrets.json', 'r') as f:
    client_config = json.load(f)

redirect_uri = 'http://localhost:8080/'
if 'installed' in client_config and 'redirect_uris' in client_config['installed']:
    redirect_uri = client_config['installed']['redirect_uris'][0]
elif 'web' in client_config and 'redirect_uris' in client_config['web']:
    redirect_uri = client_config['web']['redirect_uris'][0]

scopes = ['https://www.googleapis.com/auth/youtube.upload']
flow = Flow.from_client_secrets_file('client_secrets.json', scopes=scopes, redirect_uri=redirect_uri)

auth_url, _ = flow.authorization_url(prompt='consent')
print('\n=== 🔑 HƯỚNG DẪN XÁC THỰC YOUTUBE ===')
print('1. Nhấn vào link sau để mở trang đăng nhập Google:')
print(auth_url)
print('\n2. Chọn tài khoản YouTube của bạn và cấp quyền tải lên.')
print('3. Nếu Google cảnh báo ứng dụng chưa xác minh, hãy nhấn "Tiếp tục" hoặc "Nâng cao" -> "Đi tới [Tên app]".')
print('4. Cuối cùng, trình duyệt sẽ chuyển đến một trang web bị lỗi (ví dụ: không thể truy cập localhost).')
print('5. Đừng lo! Hãy copy TOÀN BỘ ĐƯỜNG DẪN (URL) trên thanh địa chỉ của trang lỗi đó và dán vào ô bên dưới.')
print('=====================================\n')

redirect_response = input('Dán toàn bộ URL vừa copy vào đây và nhấn Enter: ').strip()
flow.fetch_token(authorization_response=redirect_response)
youtube_credentials = flow.credentials

# 4. Thực hiện upload lên YouTube
print(f"\n🚀 Bắt đầu tải lên YouTube: {video_title}...")
youtube = build('youtube', 'v3', credentials=youtube_credentials)

body = {
    'snippet': {
        'title': video_title,
        'description': DESCRIPTION.strip(),
        'tags': video_tags,
        'categoryId': '22'
    },
    'status': {
        'privacyStatus': PRIVACY_STATUS
    }
}

media = MediaFileUpload(FILE_PATH, chunksize=-1, resumable=True)

request = youtube.videos().insert(
    part=','.join(body.keys()),
    body=body,
    media_body=media
)

response = None
while response is None:
    status, response = request.next_chunk()
    if status:
        print(f"⏳ Đang tải lên... {int(status.progress() * 100)}%")

print(f"\n✅ HOÀN TẤT! Video của bạn đã được tải lên YouTube.")
print(f"🔗 Đường dẫn video: https://youtu.be/{response['id']}")

os.remove(FILE_PATH)